# Basetable — 2024 US Presidential Election

<!-- toc -->
## Contents
  - [Variable Overview](#variable-overview)
- [1. Time Dimension](#1-time-dimension)
- [2. Bluesky](#2-bluesky)
  - [General](#general)
  - [Sentiment](#sentiment)
  - [Network](#network)
- [3. Reddit](#3-reddit)
  - [General](#general)
  - [Sentiment](#sentiment)
- [4. Newspapers](#4-newspapers)
  - [Volume & Attention](#volume-attention)
  - [Sentiment](#sentiment)
  - [Topics](#topics)
- [5. Financials](#5-financials)
  - [Market data (daily)](#market-data-daily)
  - [Macro data (monthly)](#macro-data-monthly)
- [6. Political Data](#6-political-data)
  - [Events](#events)
  - [Polls](#polls)
- [7. Lag Features (t-1)](#7-lag-features-t-1)


## Variable Overview

| # | Variable | Type | Source | Section |
|---|----------|------|--------|---------|
| 1 | polymarket_trump_prob | FLOAT [0,1] | Polymarket | Target |
| 2 | polymarket_trump_prob_lag1 | FLOAT [0,1] | Polymarket | Autoregressive |
| 3 | days_to_election | INT | Calendar | Time |
| 4 | week_nr | INT | Calendar | Time |
| 5 | weekend | INT {0,1} | Calendar | Time |
| 6 | bsky_post_count | INT | Bluesky | General |
| 7 | bsky_trump_post_share | FLOAT [0,1] | Bluesky | General |
| 8 | bsky_reply_ratio | FLOAT [0,1] | Bluesky | General |
| 9 | bsky_repost_ratio | FLOAT [0,1] | Bluesky | General |
| 10 | bsky_trump_sentiment_avg | FLOAT [-1,1] | Bluesky | Sentiment |
| 11 | bsky_harris_sentiment_avg | FLOAT [-1,1] | Bluesky | Sentiment |
| 12 | bsky_sentiment_velocity | FLOAT | Bluesky | Sentiment |
| 13 | net_density | FLOAT [0,1] | Bluesky | Network |
| 14 | net_trump_hub_degree_avg | FLOAT | Bluesky | Network |
| 15 | net_community_homophily | FLOAT [0,1] | Bluesky | Network |
| 16 | net_bridge_post_share | FLOAT [0,1] | Bluesky | Network |
| 17 | echo_chamber_velocity | FLOAT [0,1] | Bluesky | Network |
| 18 | net_modularity_Q | FLOAT | Bluesky | Network |
| 19 | reddit_post_count | INT | Reddit | General |
| 20 | reddit_trump_post_share | FLOAT [0,1] | Reddit | General |
| 21 | reddit_trump_upvote_ratio | FLOAT [0,1] | Reddit | General |
| 22 | reddit_comment_count | INT | Reddit | General |
| 23 | reddit_conservative_activity | FLOAT [0,1] | Reddit | General |
| 24 | reddit_cross_sub_ratio | FLOAT [0,1] | Reddit | General |
| 25 | reddit_trump_sentiment_avg | FLOAT [-1,1] | Reddit | Sentiment |
| 26 | reddit_sentiment_platform_gap | FLOAT | Reddit | Sentiment |
| 27 | news_title_trump_count | INT | MediaCloud | Volume |
| 28 | news_attention_asymmetry | INT | MediaCloud | Volume |
| 29 | news_trump_sentiment_avg | FLOAT [-1,1] | MediaCloud | Sentiment |
| 30 | news_trump_neg_ratio | FLOAT [0,1] | MediaCloud | Sentiment |
| 31 | news_harris_sentiment_avg | FLOAT [-1,1] | MediaCloud | Sentiment |
| 32 | news_harris_neg_ratio | FLOAT [0,1] | MediaCloud | Sentiment |
| 33 | topic_economy_share | FLOAT [0,1] | MediaCloud | Topics |
| 34 | topic_immigration_share | FLOAT [0,1] | MediaCloud | Topics |
| 35 | sp500_close | FLOAT | Yahoo Finance | Market |
| 36 | sp500_return_1d | FLOAT | Yahoo Finance | Market |
| 37 | oil_close | FLOAT | Yahoo Finance | Market |
| 38 | vix_close | FLOAT | Yahoo Finance | Market |
| 39 | bond_10y | FLOAT | FRED | Market |
| 40 | usd_index | FLOAT | Yahoo Finance | Market |
| 41 | macro_real_income | FLOAT | FRED | Macro |
| 42 | macro_real_gdp | FLOAT | FRED | Macro |
| 43 | macro_unemployment | FLOAT | BLS | Macro |
| 44 | macro_cpi | FLOAT | BLS | Macro |
| 45 | macro_consumer_sentiment | FLOAT | Michigan | Macro |
| 46 | poll_trump_avg | FLOAT | Wikipedia polls | Polls |
| 47 | poll_harris_avg | FLOAT | Wikipedia polls | Polls |
| 48 | poll_margin | FLOAT | Wikipedia polls | Polls |
| 49 | poll_n_today | INT | Wikipedia polls | Polls |
| 50 | poll_trump_7d_avg | FLOAT | Wikipedia polls | Polls |
| 51 | debate_day | INT {0,1} | Calendar | Events |
| 52 | debate_day_lag1 | INT {0,1} | Calendar | Events |
| 53 | debate_day_lag2 | INT {0,1} | Calendar | Events |
| 54 | event_assassination1 | INT {0,1} | Calendar | Events |
| 55 | event_assassination1_lag1 | INT {0,1} | Calendar | Events |
| 56 | event_biden_out | INT {0,1} | Calendar | Events |
| 57 | event_biden_out_lag1 | INT {0,1} | Calendar | Events |
| 58 | event_harris_nom | INT {0,1} | Calendar | Events |
| 59 | event_harris_nom_lag1 | INT {0,1} | Calendar | Events |
| 60 | event_assassination2 | INT {0,1} | Calendar | Events |
| 61 | event_assassination2_lag1 | INT {0,1} | Calendar | Events |
| 62 | bsky_trump_post_share_lag1 | FLOAT | Bluesky | Lags |
| 63 | bsky_trump_sentiment_avg_lag1 | FLOAT | Bluesky | Lags |
| 64 | bsky_post_count_lag1 | INT | Bluesky | Lags |
| 65 | news_title_trump_count_lag1 | INT | MediaCloud | Lags |
| 66 | news_attention_asymmetry_lag1 | INT | MediaCloud | Lags |
| 67 | news_trump_sentiment_avg_lag1 | FLOAT | MediaCloud | Lags |
| 68 | vix_close_lag1 | FLOAT | Yahoo Finance | Lags |
| 69 | sp500_return_lag1 | FLOAT | Yahoo Finance | Lags |

In [1]:
import pandas as pd
import numpy as np

# ─── Load Polymarket win probabilities as the base of the basetable ───
polymarket = pd.read_csv("../Data/1_Bronze/Polymarket/polymarket_win_probabilities.csv", parse_dates=["date"])

# The Polymarket scrape runs at ~00:03 UTC each day.
# A scrape on 2024-07-14 00:03 captures the market price at the END of 2024-07-13.
# We therefore subtract 1 day from the scraped date so the price is correctly
# aligned with the calendar day it represents.
# Example: raw date 2024-07-14 → relabeled 2024-07-13 (end-of-day price for July 13).
polymarket["date"] = polymarket["date"] - pd.Timedelta(days=1)

# Keep only Trump probability — this is the dependent variable
basetable = polymarket[["date", "Trump (%)"]].rename(columns={"Trump (%)": "polymarket_trump_prob"})
basetable["polymarket_trump_prob"] = basetable["polymarket_trump_prob"] / 100  # scale to [0, 1]

basetable = basetable.sort_values("date").reset_index(drop=True)

# Lag-1: yesterday's probability (autoregressive feature)
basetable["polymarket_trump_prob_lag1"] = basetable["polymarket_trump_prob"].shift(1)

print(f"Base: {len(basetable)} rows from {basetable['date'].min().date()} to {basetable['date'].max().date()}")
basetable.head()

Base: 124 rows from 2024-07-04 to 2024-11-04


,date,polymarket_trump_prob,polymarket_trump_prob_lag1
0,2024-07-04,0.605,NaN
1,2024-07-05,0.625,0.605
2,2024-07-06,0.625,0.625
3,2024-07-07,0.605,0.625
4,2024-07-08,0.625,0.605


# 1. Time Dimension

In [2]:
ELECTION_DAY = pd.Timestamp("2024-11-05")

# days_to_election: positive = before election, negative = after
basetable["days_to_election"] = (ELECTION_DAY - basetable["date"]).dt.days

# ISO week number
basetable["week_nr"] = basetable["date"].dt.isocalendar().week.astype(int)

# Weekend dummy (Saturday=5, Sunday=6)
basetable["weekend"] = basetable["date"].dt.dayofweek.isin([5, 6]).astype(int)

basetable[["date", "days_to_election", "week_nr", "weekend"]].head(10)

,date,days_to_election,week_nr,weekend
0,2024-07-04,124,27,0
1,2024-07-05,123,27,0
2,2024-07-06,122,27,1
3,2024-07-07,121,27,1
4,2024-07-08,120,28,0
5,2024-07-09,119,28,0
6,2024-07-10,118,28,0
7,2024-07-11,117,28,0
8,2024-07-12,116,28,0
9,2024-07-13,115,28,1


# 2. Bluesky

## General

In [3]:
# ─── Load raw Bluesky posts ───
bsky_raw = pd.read_csv("../Data/2_Silver/Bluesky/bsky_US_2024_posts.csv")
bsky_raw["timestamp"] = pd.to_datetime(bsky_raw["timestamp"], format="ISO8601", utc=True)
bsky_raw["date"] = bsky_raw["timestamp"].dt.normalize().dt.tz_localize(None)

# Filter to basetable date range
bsky_raw = bsky_raw[(bsky_raw["date"] >= basetable["date"].min()) &
                    (bsky_raw["date"] <= basetable["date"].max())]

# Trump mention flag via candidate column
bsky_raw["mentions_trump"] = bsky_raw["candidate"] == "CandidateA"

# ─── Aggregate per day ───
bsky_daily = bsky_raw.groupby("date").agg(
    bsky_post_count       = ("uri",           "count"),
    bsky_trump_post_share = ("mentions_trump", "mean"),
    bsky_reply_ratio      = ("is_reply",       "mean"),
    bsky_repost_ratio     = ("reposts",        lambda x: (x > 0).mean()),
).reset_index()

# Merge into basetable
basetable = basetable.merge(bsky_daily, on="date", how="left")

basetable[["date", "bsky_post_count", "bsky_trump_post_share", "bsky_reply_ratio", "bsky_repost_ratio"]].head(10)

,date,bsky_post_count,bsky_trump_post_share,bsky_reply_ratio,bsky_repost_ratio
0,2024-07-04,NaN,NaN,NaN,NaN
1,2024-07-05,36.0,0.361111,0.472222,0.222222
2,2024-07-06,23.0,0.434783,0.130435,0.130435
3,2024-07-07,27.0,0.481481,0.296296,0.148148
4,2024-07-08,22.0,0.590909,0.000000,0.454545
5,2024-07-09,33.0,0.454545,0.181818,0.181818
6,2024-07-10,39.0,0.461538,0.205128,0.179487
7,2024-07-11,62.0,0.354839,0.032258,0.112903
8,2024-07-12,43.0,0.372093,0.139535,0.139535
9,2024-07-13,32.0,0.375000,0.187500,0.250000


## Sentiment

In [4]:
# ─── Bluesky Sentiment — VADER ─────────────────────────────────────────────
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

sia = SentimentIntensityAnalyzer()

# Compound score in [-1, 1] per post
bsky_raw["vader"] = bsky_raw["text"].astype(str).apply(
    lambda t: sia.polarity_scores(t)["compound"]
)

# Strong sentiment flags: compound > 0.4 = clearly positive, < -0.4 = clearly negative
bsky_raw["is_strong_pos"] = bsky_raw["vader"] > 0.4
bsky_raw["is_strong_neg"] = bsky_raw["vader"] < -0.4

# Per-day aggregate per candidate: mean, std (controversy), strong-pos/neg share
sent_daily_agg = (
    bsky_raw.groupby(["date", "candidate"]).agg(
        vader_mean    =("vader",         "mean"),
        vader_std     =("vader",         "std"),
        strong_pos    =("is_strong_pos", "mean"),
        strong_neg    =("is_strong_neg", "mean"),
    )
    .unstack(fill_value=np.nan)
)
# Flatten multi-level column index produced by unstack
sent_daily_agg.columns = ["__".join(col).strip() for col in sent_daily_agg.columns.values]
sent_daily_agg = sent_daily_agg.reset_index()

# Map generic CandidateA/B names to trump/harris column names
rename_map = {
    "vader_mean__CandidateA":  "bsky_trump_sentiment_avg",
    "vader_mean__CandidateB":  "bsky_harris_sentiment_avg",
    "vader_std__CandidateA":   "bsky_trump_sent_std",
    "vader_std__CandidateB":   "bsky_harris_sent_std",
    "strong_pos__CandidateA":  "bsky_trump_strong_pos_ratio",
    "strong_pos__CandidateB":  "bsky_harris_strong_pos_ratio",
    "strong_neg__CandidateA":  "bsky_trump_strong_neg_ratio",
    "strong_neg__CandidateB":  "bsky_harris_strong_neg_ratio",
}
sent_daily_agg = sent_daily_agg.rename(columns=rename_map)

NEW_BSKY_SENT = [
    "bsky_trump_sentiment_avg",    "bsky_harris_sentiment_avg",
    "bsky_trump_sent_std",         "bsky_harris_sent_std",
    "bsky_trump_strong_pos_ratio", "bsky_harris_strong_pos_ratio",
    "bsky_trump_strong_neg_ratio", "bsky_harris_strong_neg_ratio",
]
for col in NEW_BSKY_SENT:
    if col not in sent_daily_agg.columns:
        sent_daily_agg[col] = np.nan

basetable = basetable.merge(sent_daily_agg[["date"] + NEW_BSKY_SENT], on="date", how="left")

# Sentiment velocity: day-over-day change in Trump compound score
basetable["bsky_sentiment_velocity"] = basetable["bsky_trump_sentiment_avg"].diff()

print(f"Sentiment scored: {len(bsky_raw):,} posts")
basetable[["date", "bsky_trump_sentiment_avg", "bsky_harris_sentiment_avg",
           "bsky_trump_sent_std", "bsky_trump_strong_pos_ratio", "bsky_trump_strong_neg_ratio"]].dropna().head(8)

Sentiment scored: 27,215 posts


,date,bsky_trump_sentiment_avg,bsky_harris_sentiment_avg,bsky_trump_sent_std,bsky_trump_strong_pos_ratio,bsky_trump_strong_neg_ratio
1,2024-07-05,-0.096700,0.259244,0.475500,0.153846,0.307692
2,2024-07-06,0.094200,0.050900,0.534687,0.400000,0.200000
3,2024-07-07,-0.297446,0.205490,0.326570,0.000000,0.384615
4,2024-07-08,-0.238692,0.032386,0.381102,0.076923,0.384615
5,2024-07-09,-0.151707,0.197820,0.440960,0.133333,0.400000
6,2024-07-10,0.162794,0.179015,0.666952,0.555556,0.277778
7,2024-07-11,-0.077805,0.044718,0.531273,0.136364,0.363636
8,2024-07-12,-0.058800,0.072950,0.473307,0.187500,0.375000


## Network

In [5]:
# ─── Bluesky Network features — computed daily with NetworkX ─────────────────
import networkx as nx
from networkx.algorithms.community import greedy_modularity_communities
from networkx.algorithms.community.quality import modularity as nx_modularity

# ── Bridge nodes: load pre-computed global centrality labels ──────────────────
centrality = pd.read_csv("../Data/2_Silver/Bluesky/bsky_US_2024_centrality.csv")
bridge_nodes = set(centrality.loc[centrality["is_bridge"], "node"])

# ── Normalize author handles: strip domain suffix (e.g. "user.bsky.social" → "user") ──
# centrality["node"] contains handles only; bsky_raw["author"] has full domain
bsky_raw["author_handle"] = bsky_raw["author"].str.split(".").str[0]

# ── Resolve parent_uri → parent author handle (within our dataset) ──────────
uri_to_handle    = dict(zip(bsky_raw["uri"], bsky_raw["author_handle"]))
uri_to_candidate = dict(zip(bsky_raw["uri"], bsky_raw["candidate"]))
bsky_raw["parent_handle"] = bsky_raw["parent_uri"].map(uri_to_handle)

# ── Each author's predominant candidate affiliation (by handle) ───────────────
author_camp = (
    bsky_raw.groupby("author_handle")["candidate"]
    .agg(lambda s: s.value_counts().idxmax())
)

# ── Day-by-day network metrics ────────────────────────────────────────────────
NET_COLS = ["net_density", "net_trump_hub_degree_avg", "net_community_homophily",
            "net_bridge_post_share", "echo_chamber_velocity", "net_modularity_Q"]

records = []
prev_trump_users = set()

for day, day_df in bsky_raw.groupby("date"):
    row = {"date": day}

    # Reply edges: replier handle → parent handle (both must be known)
    edges = day_df.loc[
        day_df["is_reply"] & day_df["parent_handle"].notna(),
        ["author_handle", "parent_handle"]
    ]

    if len(edges) < 5:
        row.update({c: np.nan for c in NET_COLS})
        records.append(row)
        prev_trump_users = set()
        continue

    G = nx.DiGraph()
    G.add_edges_from(zip(edges["author_handle"], edges["parent_handle"]))

    # 1. Density
    row["net_density"] = nx.density(G)

    # 2. Top-10 Trump hub degree (mean in-degree of top-10 CandidateA nodes)
    trump_nodes = {n for n in G.nodes() if author_camp.get(n, "") == "CandidateA"}
    trump_indeg  = sorted((G.in_degree(n) for n in trump_nodes), reverse=True)[:10]
    row["net_trump_hub_degree_avg"] = np.mean(trump_indeg) if trump_indeg else np.nan

    # 3. Community homophily (dyadicity): fraction of edges within same camp
    same = sum(1 for u, v in G.edges()
               if author_camp.get(u, "") == author_camp.get(v, "") != "")
    row["net_community_homophily"] = same / G.number_of_edges()

    # 4. Bridge post share: share of today's authors (by handle) that are global bridge nodes
    today_handles = set(day_df["author_handle"])
    row["net_bridge_post_share"] = (
        len(today_handles & bridge_nodes) / len(today_handles)
        if today_handles else np.nan
    )

    # 5. Echo chamber velocity: fraction of Trump users new to the community today
    today_trump = set(day_df.loc[day_df["candidate"] == "CandidateA", "author_handle"])
    new_entrants = today_trump - prev_trump_users
    row["echo_chamber_velocity"] = (
        len(new_entrants) / len(today_trump) if today_trump else np.nan
    )
    prev_trump_users = today_trump

    # 6. Modularity Q (undirected, greedy community detection)
    G_und = G.to_undirected()
    try:
        comms = list(greedy_modularity_communities(G_und))
        row["net_modularity_Q"] = nx_modularity(G_und, comms)
    except Exception:
        row["net_modularity_Q"] = np.nan

    records.append(row)

net_df = pd.DataFrame(records)
basetable = basetable.merge(net_df, on="date", how="left")

print(f"Network features computed for {len(net_df)} days")
print(f"Bridge nodes in centrality file: {len(bridge_nodes)}")
basetable[["date"] + NET_COLS].dropna(subset=["net_density"]).head(8)

Network features computed for 123 days
Bridge nodes in centrality file: 47


,date,net_density,net_trump_hub_degree_avg,net_community_homophily,net_bridge_post_share,echo_chamber_velocity,net_modularity_Q
1,2024-07-05,0.070513,1.000000,0.363636,0.000000,1.000000,0.661157
3,2024-07-07,0.333333,1.000000,0.500000,0.062500,1.000000,0.666667
8,2024-07-12,0.069444,1.000000,0.200000,0.000000,1.000000,0.720000
10,2024-07-14,0.052381,1.555556,0.227273,0.000000,1.000000,0.759766
11,2024-07-15,0.054945,1.000000,0.500000,0.000000,0.677419,0.843750
12,2024-07-16,0.083333,1.000000,0.333333,0.000000,0.750000,0.500000
13,2024-07-17,0.075758,0.800000,0.000000,0.000000,0.785714,0.833333
14,2024-07-18,0.036842,1.500000,0.285714,0.015625,0.755556,0.716837


# 3. Reddit

## General

In [6]:
# ─── Reddit — Data loading & General features ───────────────────────────────
# We load posts from ALL 7 subreddits so that volume metrics and sentiment
# capture the full breadth of political discussion on Reddit:
#   r/politics, r/conservative, r/worldnews  (news-oriented)
#   r/democrats, r/republican, r/liberal      (partisan)
#   r/trump                                   (candidate-specific)
#
# For comments we skip the two largest files (r/politics 3.6 GB and
# r/worldnews 3.1 GB) to keep loading time under ~5 minutes; the five
# smaller subreddits still give us >2 M comments to work with.
#
# Both Trump AND Harris mention flags are created from the start so that
# all downstream sentiment cells can use the same filtered DataFrames.
import json as _json, os as _os

def _load_posts(path, fields):
    rows = []
    with open(path, encoding='utf-8') as fh:
        for line in fh:
            d = _json.loads(line)
            rows.append({k: d.get(k) for k in fields})
    df = pd.DataFrame(rows)
    df['date'] = pd.to_datetime(
        df['created_utc'].fillna(df.get('created', pd.NaT)),
        unit='s', utc=True).dt.normalize().dt.tz_localize(None)
    return df

def _load_comments(path):
    # Only keep the fields we actually use — keeps memory low on large files.
    rows = []
    with open(path, encoding='utf-8') as fh:
        for line in fh:
            d = _json.loads(line)
            body = d.get('body', '') or ''
            rows.append({
                'author':      d.get('author'),
                'body':        body,
                'subreddit':   d.get('subreddit', ''),
                'created_utc': d.get('created_utc') or d.get('created'),
                'mentions_trump':  'trump'  in body.lower(),
                'mentions_harris': 'harris' in body.lower() or 'kamala' in body.lower(),
            })
    df = pd.DataFrame(rows)
    df['date'] = pd.to_datetime(
        df['created_utc'], unit='s', utc=True).dt.normalize().dt.tz_localize(None)
    return df

POST_FIELDS = ['author', 'subreddit', 'title', 'upvote_ratio',
               'num_comments', 'score', 'created_utc', 'created']

# --- Posts: all 7 subreddits -------------------------------------------------
POST_SUBS = ['politics', 'conservative', 'worldnews',
             'democrats', 'trump', 'republican', 'liberal']
post_dfs = []
for _sub in POST_SUBS:
    _path = f'../Data/1_Bronze/Reddit/r_{_sub}_posts.jsonl'
    if _os.path.exists(_path):
        post_dfs.append(_load_posts(_path, POST_FIELDS))
        print(f'  Posts loaded: r/{_sub}')
    else:
        print(f'  MISSING: {_path}')
all_posts = pd.concat(post_dfs, ignore_index=True)

# Candidate mention flags on post titles
title_lower = all_posts['title'].str.lower().fillna('')
all_posts['mentions_trump']  = title_lower.str.contains('trump|donald|maga',  na=False)
all_posts['mentions_harris'] = title_lower.str.contains('harris|kamala|walz', na=False)

# --- Comments: 5 smaller subreddits (skip politics 3.6 GB, worldnews 3.1 GB) -
COMMENT_SUBS = ['conservative', 'trump', 'republican', 'liberal', 'democrats']
comm_dfs = []
for _sub in COMMENT_SUBS:
    _path = f'../Data/1_Bronze/Reddit/r_{_sub}_comments.jsonl'
    if _os.path.exists(_path):
        comm_dfs.append(_load_comments(_path))
        print(f'  Comments loaded: r/{_sub}')
    else:
        print(f'  MISSING: {_path}')
comments_df = pd.concat(comm_dfs, ignore_index=True)

# Filter everything to the basetable date range
date_min, date_max = basetable['date'].min(), basetable['date'].max()
all_posts   = all_posts[  (all_posts['date']   >= date_min) & (all_posts['date']   <= date_max)].copy()
comments_df = comments_df[(comments_df['date'] >= date_min) & (comments_df['date'] <= date_max)].copy()

# ─── Volume / engagement features ────────────────────────────────────────────
# 1. reddit_post_count  – total daily posts across all subreddits
post_count = all_posts.groupby('date').size().rename('reddit_post_count').reset_index()

# 2. reddit_trump_post_share  – fraction of posts mentioning Trump
trump_share = (all_posts.groupby('date')['mentions_trump']
               .mean().rename('reddit_trump_post_share').reset_index())

# 3. reddit_harris_post_share – fraction of posts mentioning Harris (NEW)
harris_share = (all_posts.groupby('date')['mentions_harris']
                .mean().rename('reddit_harris_post_share').reset_index())

# 4. reddit_trump_upvote_ratio – community approval of Trump-related posts
trump_upvote = (all_posts[all_posts['mentions_trump']]
                .groupby('date')['upvote_ratio']
                .mean().rename('reddit_trump_upvote_ratio').reset_index())

# 5. reddit_comment_count – total comments linked to all posts
comment_count = (all_posts.groupby('date')['num_comments']
                 .sum().rename('reddit_comment_count').reset_index())

# 6. reddit_conservative_activity – normalised r/conservative volume (posts + comments)
_conserv_posts = all_posts[all_posts['subreddit'] == 'conservative'].groupby('date').size()
_conserv_comms = comments_df[comments_df['subreddit'].str.lower() == 'conservative'].groupby('date').size()
_conserv_raw   = _conserv_posts.add(_conserv_comms, fill_value=0)
conserv_activity = (_conserv_raw / _conserv_raw.max()).rename('reddit_conservative_activity').reset_index()

# 7. reddit_cross_sub_ratio – authors active in BOTH r/conservative AND r/politics
#    Proxy for bridge accounts that span ideological communities
cross_rows = []
for day, grp in all_posts.groupby('date'):
    c_aut = set(grp.loc[grp['subreddit'].str.lower() == 'conservative', 'author'].dropna())
    p_aut = set(grp.loc[grp['subreddit'].str.lower() == 'politics',     'author'].dropna())
    all_aut = c_aut | p_aut
    cross_rows.append({'date': day,
                       'reddit_cross_sub_ratio': len(c_aut & p_aut) / len(all_aut) if all_aut else np.nan})
cross_df = pd.DataFrame(cross_rows)

GENERAL_REDDIT = ['reddit_post_count', 'reddit_trump_post_share', 'reddit_harris_post_share',
                  'reddit_trump_upvote_ratio', 'reddit_comment_count',
                  'reddit_conservative_activity', 'reddit_cross_sub_ratio']
for _df in [post_count, trump_share, harris_share, trump_upvote,
            comment_count, conserv_activity, cross_df]:
    basetable = basetable.merge(_df, on='date', how='left')

print(f'Posts loaded: {len(all_posts):,} total across {len(POST_SUBS)} subreddits')
print(f'Comments loaded: {len(comments_df):,} total across {len(COMMENT_SUBS)} subreddits')
basetable[['date'] + GENERAL_REDDIT].dropna(subset=['reddit_post_count']).head(5)


  Posts loaded: r/politics


  Posts loaded: r/conservative


  Posts loaded: r/worldnews


  Posts loaded: r/democrats


  Posts loaded: r/trump


...

  Comments loaded: r/republican


  Comments loaded: r/liberal


  Comments loaded: r/democrats


Posts loaded: 200,068 total across 7 subreddits
Comments loaded: 1,439,714 total across 5 subreddits


,date,reddit_post_count,reddit_trump_post_share,reddit_harris_post_share,reddit_trump_upvote_ratio,reddit_comment_count,reddit_conservative_activity,reddit_cross_sub_ratio
0,2024-07-04,469.0,0.243070,0.070362,0.819386,44023.0,NaN,0.000000
1,2024-07-05,1309.0,0.200153,0.046600,0.840000,71356.0,0.146050,0.008114
2,2024-07-06,1160.0,0.189655,0.018966,0.854227,63544.0,0.131741,0.006849
3,2024-07-07,1098.0,0.154827,0.040073,0.865706,48879.0,0.150019,0.010724
4,2024-07-08,1346.0,0.163447,0.024517,0.855136,74782.0,0.120357,0.014019


## Sentiment

In [7]:
# ─── Reddit — VADER Sentiment (Trump & Harris) ───────────────────────────────
# We compute VADER compound scores for BOTH candidates using text from all
# loaded subreddits:
#   - Post titles that mention the candidate
#   - Comment bodies that mention the candidate
#
# For each candidate we compute four daily aggregates:
#   *_sentiment_avg   – mean compound score (positive = favourable tone)
#   *_sent_std        – std of compound scores (= controversy / polarisation)
#   *_strong_pos_ratio – fraction of texts with compound > +0.4 (clearly positive)
#   *_strong_neg_ratio – fraction of texts with compound < -0.4 (clearly negative)
#
# Cross-candidate gap (Trump - Harris) captures RELATIVE sentiment advantage,
# which is conceptually more predictive than absolute levels for a prediction
# market that already prices in baseline sentiment.
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

_sia = SentimentIntensityAnalyzer()


def _reddit_cand_sentiment(all_posts, comments_df, trump_flag, prefix):
    # Collect all text (titles + comment bodies) that mention the candidate.
    titles = (all_posts.loc[all_posts[trump_flag], ['date', 'title']]
              .rename(columns={'title': 'text'}))
    comms  = (comments_df.loc[comments_df[trump_flag], ['date', 'body']]
              .rename(columns={'body': 'text'}))
    cand_text = pd.concat([titles, comms], ignore_index=True)

    # VADER compound score for each piece of text
    cand_text['vader'] = cand_text['text'].astype(str).apply(
        lambda t: _sia.polarity_scores(t)['compound'])
    cand_text['is_strong_pos'] = cand_text['vader'] >  0.4
    cand_text['is_strong_neg'] = cand_text['vader'] < -0.4

    # Aggregate per day
    agg = cand_text.groupby('date').agg(
        sentiment_avg    = ('vader',         'mean'),
        sent_std         = ('vader',         'std'),
        strong_pos_ratio = ('is_strong_pos', 'mean'),
        strong_neg_ratio = ('is_strong_neg', 'mean'),
        n_texts          = ('vader',         'count'),
    ).reset_index()

    return agg.rename(columns={
        'sentiment_avg':    f'reddit_{prefix}_sentiment_avg',
        'sent_std':         f'reddit_{prefix}_sent_std',
        'strong_pos_ratio': f'reddit_{prefix}_strong_pos_ratio',
        'strong_neg_ratio': f'reddit_{prefix}_strong_neg_ratio',
        'n_texts':          f'reddit_{prefix}_n_texts',
    }), cand_text


trump_sent_df,  trump_text  = _reddit_cand_sentiment(all_posts, comments_df, 'mentions_trump',  'trump')
harris_sent_df, harris_text = _reddit_cand_sentiment(all_posts, comments_df, 'mentions_harris', 'harris')

for _df in [trump_sent_df, harris_sent_df]:
    basetable = basetable.merge(_df, on='date', how='left')

# Cross-candidate Reddit sentiment gap
basetable['reddit_sentiment_gap'] = (
    basetable['reddit_trump_sentiment_avg'] - basetable['reddit_harris_sentiment_avg']
)

# Cross-platform gap: Reddit vs Bluesky Trump sentiment
# A large positive value = Reddit is more pro-Trump than Bluesky.
# This can signal platform-specific echo chambers.
basetable['reddit_sentiment_platform_gap'] = (
    basetable['reddit_trump_sentiment_avg'] - basetable['bsky_trump_sentiment_avg']
)

SENT_REDDIT = [
    'reddit_trump_sentiment_avg',    'reddit_trump_sent_std',
    'reddit_trump_strong_pos_ratio', 'reddit_trump_strong_neg_ratio',
    'reddit_harris_sentiment_avg',   'reddit_harris_sent_std',
    'reddit_harris_strong_pos_ratio','reddit_harris_strong_neg_ratio',
    'reddit_sentiment_gap', 'reddit_sentiment_platform_gap',
]
print(f'Trump texts:  {len(trump_text):,}')
print(f'Harris texts: {len(harris_text):,}')
basetable[['date'] + SENT_REDDIT].dropna(subset=['reddit_trump_sentiment_avg']).head(5)


Trump texts:  192,753
Harris texts: 104,995


,date,reddit_trump_sentiment_avg,reddit_trump_sent_std,reddit_trump_strong_pos_ratio,reddit_trump_strong_neg_ratio,reddit_harris_sentiment_avg,reddit_harris_sent_std,reddit_harris_strong_pos_ratio,reddit_harris_strong_neg_ratio,reddit_sentiment_gap,reddit_sentiment_platform_gap
0,2024-07-04,-0.066882,0.378345,0.140351,0.201754,0.053845,0.408491,0.181818,0.212121,-0.120727,NaN
1,2024-07-05,0.016446,0.559151,0.281930,0.274361,0.064089,0.477395,0.289855,0.183575,-0.047644,0.113146
2,2024-07-06,0.018668,0.576016,0.295406,0.295406,0.154418,0.529493,0.371681,0.194690,-0.135750,-0.075532
3,2024-07-07,0.027416,0.553370,0.287298,0.277218,0.040112,0.506326,0.333333,0.260000,-0.012696,0.324862
4,2024-07-08,0.012438,0.602118,0.320378,0.310924,0.022687,0.546934,0.320856,0.288770,-0.010249,0.251131


# 4. Newspapers

## Volume & Attention

In [8]:
# ─── Newspapers — Data loading & Volume features ─────────────────────────────
mc_daily   = pd.read_csv("../Data/1_Bronze/Newspapers/mediacloud_daily.csv",
                          parse_dates=["date"])
mc_stories = pd.read_csv("../Data/1_Bronze/Newspapers/mediacloud_stories.csv",
                          parse_dates=["date"])

# Volume metrics directly from pre-aggregated daily counts
news_volume = mc_daily[["date", "trump", "harris"]].copy()
news_volume["news_title_trump_count"]   = news_volume["trump"]
news_volume["news_attention_asymmetry"] = news_volume["trump"] - news_volume["harris"]

basetable = basetable.merge(
    news_volume[["date", "news_title_trump_count", "news_attention_asymmetry"]],
    on="date", how="left")

VOL_NEWS = ["news_title_trump_count", "news_attention_asymmetry"]
print(f'MediaCloud daily: {len(mc_daily)} days  |  Stories: {len(mc_stories):,}')
basetable[["date"] + VOL_NEWS].dropna().head(5)

MediaCloud daily: 123 days  |  Stories: 235,191


,date,news_title_trump_count,news_attention_asymmetry
1,2024-07-05,851.0,592.0
2,2024-07-06,456.0,346.0
3,2024-07-07,421.0,230.0
4,2024-07-08,921.0,656.0
5,2024-07-09,1070.0,777.0


## Sentiment

In [9]:
# ─── Newspapers — Sentiment ─────────────────────────────────────────────────────
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

sia_news = SentimentIntensityAnalyzer()

mc_stories["vader"] = mc_stories["title"].astype(str).apply(
    lambda t: sia_news.polarity_scores(t)["compound"])

# Threshold flags
mc_stories["is_negative"]   = mc_stories["vader"] < -0.05  # any negative tone
mc_stories["is_strong_pos"] = mc_stories["vader"] >  0.4   # clearly positive
mc_stories["is_strong_neg"] = mc_stories["vader"] < -0.4   # clearly negative

def _news_sentiment(df, query_label, prefix):
    """Aggregate VADER sentiment for a single query (trump or harris) per day."""
    sub = df[df["query"] == query_label].groupby("date").agg(
        sentiment_avg = ("vader",         "mean"),
        sent_std      = ("vader",         "std"),   # dispersion / controversy
        neg_ratio     = ("is_negative",   "mean"),
        strong_pos    = ("is_strong_pos", "mean"),  # fraction strongly positive
        strong_neg    = ("is_strong_neg", "mean"),  # fraction strongly negative
    ).reset_index()
    return sub.rename(columns={
        "sentiment_avg": f"news_{prefix}_sentiment_avg",
        "sent_std":      f"news_{prefix}_sent_std",
        "neg_ratio":     f"news_{prefix}_neg_ratio",
        "strong_pos":    f"news_{prefix}_strong_pos_ratio",
        "strong_neg":    f"news_{prefix}_strong_neg_ratio",
    })

trump_sent_news  = _news_sentiment(mc_stories, "trump",  "trump")
harris_sent_news = _news_sentiment(mc_stories, "harris", "harris")

for df_feat in [trump_sent_news, harris_sent_news]:
    basetable = basetable.merge(df_feat, on="date", how="left")

SENT_NEWS = [
    "news_trump_sentiment_avg",    "news_trump_sent_std",
    "news_trump_neg_ratio",        "news_trump_strong_pos_ratio",  "news_trump_strong_neg_ratio",
    "news_harris_sentiment_avg",   "news_harris_sent_std",
    "news_harris_neg_ratio",       "news_harris_strong_pos_ratio", "news_harris_strong_neg_ratio",
]
print(f"VADER scored: {len(mc_stories):,} story titles")
basetable[["date"] + SENT_NEWS].dropna().head(5)

VADER scored: 235,191 story titles


,date,news_trump_sentiment_avg,news_trump_sent_std,news_trump_neg_ratio,news_trump_strong_pos_ratio,news_trump_strong_neg_ratio,news_harris_sentiment_avg,news_harris_sent_std,news_harris_neg_ratio,news_harris_strong_pos_ratio,news_harris_strong_neg_ratio
1,2024-07-05,-0.036589,0.391865,0.356052,0.183314,0.196240,-0.016054,0.361147,0.305019,0.177606,0.162162
2,2024-07-06,-0.074378,0.369174,0.396930,0.135965,0.214912,-0.083504,0.334140,0.372727,0.081818,0.163636
3,2024-07-07,-0.038358,0.351479,0.349169,0.152019,0.197150,-0.027279,0.313849,0.298429,0.130890,0.157068
4,2024-07-08,-0.040297,0.358302,0.332248,0.138979,0.184582,-0.027917,0.351952,0.305660,0.128302,0.158491
5,2024-07-09,-0.031364,0.351684,0.330841,0.140187,0.177570,-0.003810,0.348932,0.307167,0.170648,0.163823


In [10]:
# ─── Newspapers — FinBERT Sentiment ─────────────────────────────────────────────────
# FinBERT (ProsusAI/finbert) understands context unlike rule-based VADER.
# We keep VADER features above AND add FinBERT features here (both used).
#
# Caching: results are saved to Data/3_Gold/finbert_news_cache.csv on first run.
# Subsequent runs load from cache instantly. Delete the file to recompute.
import os
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
import warnings
warnings.filterwarnings('ignore')

CACHE_PATH = '../Data/3_Gold/finbert_news_cache.csv'

if os.path.exists(CACHE_PATH):
    # Fast path: load pre-computed scores
    print(f'Loading FinBERT cache from {CACHE_PATH}')
    sampled_fb = pd.read_csv(CACHE_PATH, parse_dates=['date'])
    print(f'  Loaded {len(sampled_fb):,} rows')

else:
    # Slow path: run FinBERT (takes ~5 min on CPU, then caches)
    from transformers import pipeline

    print('Loading FinBERT model (ProsusAI/finbert)...')
    finbert_pipe = pipeline(
        'text-classification',
        model='ProsusAI/finbert',
        top_k=None,
        device=-1,  # CPU
    )

    mc_sub = mc_stories[mc_stories['query'].isin(['trump', 'harris'])].copy()
    sampled_fb = (
        mc_sub
        .groupby(['date', 'query'], group_keys=False)
        .apply(lambda x: x.sample(min(len(x), 10), random_state=42))
        .reset_index(drop=True)
    )
    print(f'Running FinBERT on {len(sampled_fb):,} headlines...')

    texts_fb = sampled_fb['title'].astype(str).tolist()
    BATCH_FB = 64
    fb_results = []
    for i in range(0, len(texts_fb), BATCH_FB):
        batch_out = finbert_pipe(texts_fb[i:i+BATCH_FB], truncation=True, max_length=128)
        for scores in batch_out:
            d = {s['label']: s['score'] for s in scores}
            fb_results.append(d)

    res_fb = pd.DataFrame(fb_results)
    sampled_fb = sampled_fb.copy()
    sampled_fb['finbert_pos']  = res_fb['positive'].values
    sampled_fb['finbert_neg']  = res_fb['negative'].values
    sampled_fb['finbert_neut'] = res_fb['neutral'].values
    sampled_fb['finbert_net']  = sampled_fb['finbert_pos'] - sampled_fb['finbert_neg']

    # Save cache
    sampled_fb.to_csv(CACHE_PATH, index=False)
    print(f'Cache saved to {CACHE_PATH}')


def _finbert_agg(df, query_label, prefix):
    sub = df[df['query'] == query_label].groupby('date').agg(
        finbert_avg       = ('finbert_net', 'mean'),
        finbert_std       = ('finbert_net', 'std'),
        finbert_pos_ratio = ('finbert_pos', 'mean'),
        finbert_neg_ratio = ('finbert_neg', 'mean'),
    ).reset_index()
    return sub.rename(columns={
        'finbert_avg':       f'news_{prefix}_finbert_avg',
        'finbert_std':       f'news_{prefix}_finbert_std',
        'finbert_pos_ratio': f'news_{prefix}_finbert_pos_ratio',
        'finbert_neg_ratio': f'news_{prefix}_finbert_neg_ratio',
    })


trump_finbert  = _finbert_agg(sampled_fb, 'trump',  'trump')
harris_finbert = _finbert_agg(sampled_fb, 'harris', 'harris')

for df_feat in [trump_finbert, harris_finbert]:
    basetable = basetable.merge(df_feat, on='date', how='left')

basetable['news_finbert_gap'] = (
    basetable['news_trump_finbert_avg'] - basetable['news_harris_finbert_avg']
)

SENT_NEWS_FINBERT = [
    'news_trump_finbert_avg',        'news_trump_finbert_std',
    'news_trump_finbert_pos_ratio',  'news_trump_finbert_neg_ratio',
    'news_harris_finbert_avg',       'news_harris_finbert_std',
    'news_harris_finbert_pos_ratio', 'news_harris_finbert_neg_ratio',
    'news_finbert_gap',
]
print(f'\nFinBERT features added: {len(SENT_NEWS_FINBERT)}')
basetable[['date'] + SENT_NEWS_FINBERT].dropna().head(5)


Loading FinBERT cache from ../Data/3_Gold/finbert_news_cache.csv
  Loaded 2,290 rows

FinBERT features added: 9


,date,news_trump_finbert_avg,news_trump_finbert_std,news_trump_finbert_pos_ratio,news_trump_finbert_neg_ratio,news_harris_finbert_avg,news_harris_finbert_std,news_harris_finbert_pos_ratio,news_harris_finbert_neg_ratio,news_finbert_gap
1,2024-07-05,-0.150967,0.313397,0.125762,0.276730,-0.398398,0.372030,0.056038,0.454435,0.247430
2,2024-07-06,-0.201044,0.376261,0.084041,0.285085,-0.280045,0.338235,0.038091,0.318136,0.079002
3,2024-07-07,-0.137779,0.333285,0.075267,0.213046,-0.045340,0.155267,0.077965,0.123304,-0.092439
4,2024-07-08,-0.092228,0.258520,0.080918,0.173145,-0.220645,0.351204,0.084141,0.304786,0.128417
5,2024-07-09,-0.425211,0.409766,0.049314,0.474525,-0.090029,0.453677,0.147755,0.237783,-0.335182


## Topics

In [11]:
# ─── Newspapers — Topics (keyword matching) ──────────────────────────────────
ECONOMY_KW = (r"\b(inflation|economy|economic|gdp|unemployment|recession|"
              r"jobs?|wages?|trade|tariff|tax|debt|deficit|interest rate|"
              r"federal reserve|fed |cost of living|prices?|spending)\b")
IMMIGRATION_KW = (r"\b(immigra|immigrant|border|migrant|asylum|deport|"
                  r"illegal alien|undocumented|daca|visa|sanctuary|wall)\b")

# Deduplicate by url per day — avoid counting stories twice across queries
stories_dedup = mc_stories.drop_duplicates(subset=["date", "url"]).copy()
stories_dedup["title_lower"] = stories_dedup["title"].str.lower()

stories_dedup["topic_economy"]     = stories_dedup["title_lower"].str.contains(
    ECONOMY_KW, regex=True, na=False)
stories_dedup["topic_immigration"] = stories_dedup["title_lower"].str.contains(
    IMMIGRATION_KW, regex=True, na=False)

topic_daily = stories_dedup.groupby("date").agg(
    total       = ("url",               "count"),
    econ_count  = ("topic_economy",     "sum"),
    immig_count = ("topic_immigration", "sum"),
).reset_index()
topic_daily["topic_economy_share"]     = topic_daily["econ_count"]  / topic_daily["total"]
topic_daily["topic_immigration_share"] = topic_daily["immig_count"] / topic_daily["total"]

basetable = basetable.merge(
    topic_daily[["date", "topic_economy_share", "topic_immigration_share"]],
    on="date", how="left")

TOPIC_COLS = ["topic_economy_share", "topic_immigration_share"]
print(f"Deduplicated stories: {len(stories_dedup):,}")
basetable[["date"] + TOPIC_COLS].dropna().head(5)

Deduplicated stories: 148,877


,date,topic_economy_share,topic_immigration_share
1,2024-07-05,0.030702,0.007675
2,2024-07-06,0.010373,0.004149
3,2024-07-07,0.021882,0.015317
4,2024-07-08,0.027721,0.011294
5,2024-07-09,0.032286,0.008726


# 5. Financials

## Market data (daily)
Weekends and holidays have no trading data → forward-fill from last known trading day.
Daily return computed as percentage change vs previous close.

In [12]:
# ─── Load market data ───
market = pd.read_csv("../Data/1_Bronze/Financials/market.csv", parse_dates=["Date"])
market = market.rename(columns={
    "Date":        "date",
    "SP500":       "sp500_close",
    "Oil":         "oil_close",
    "VIX":         "vix_close",
    "TenYearBond": "bond_10y",
    "USDIndex":    "usd_index",
})
market = market.sort_values("date").reset_index(drop=True)

# ─── All features computed on trading-day data (before ffill) ───
# This prevents spurious zero-return weekends from polluting rolling windows / lag shifts.

# S&P 500 — Returns & Momentum
market["sp500_return_1d"]        = market["sp500_close"].pct_change(1)  * 100
market["sp500_return_3d"]        = market["sp500_close"].pct_change(3)  * 100
market["sp500_return_7d"]        = market["sp500_close"].pct_change(7)  * 100
market["sp500_return_14d"]       = market["sp500_close"].pct_change(14) * 100
market["sp500_rolling_mean_7d"]  = market["sp500_return_1d"].rolling(7).mean()
market["sp500_rolling_mean_14d"] = market["sp500_return_1d"].rolling(14).mean()

# S&P 500 — Volatility
market["sp500_vol_7d"]   = market["sp500_return_1d"].rolling(7).std()
market["sp500_vol_14d"]  = market["sp500_return_1d"].rolling(14).std()
market["sp500_vol_30d"]  = market["sp500_return_1d"].rolling(30).std()
market["sp500_range_7d"] = (
    (market["sp500_close"].rolling(7).max() - market["sp500_close"].rolling(7).min())
    / market["sp500_close"].rolling(7).mean()
)

# S&P 500 — Level-Based
market["sp500_dist_from_high_30d"] = (
    (market["sp500_close"] - market["sp500_close"].rolling(30).max())
    / market["sp500_close"].rolling(30).max() * 100
)
market["sp500_dist_from_low_30d"] = (
    (market["sp500_close"] - market["sp500_close"].rolling(30).min())
    / market["sp500_close"].rolling(30).min() * 100
)
market["sp500_zscore_30d"] = (
    (market["sp500_close"] - market["sp500_close"].rolling(30).mean())
    / market["sp500_close"].rolling(30).std()
)
_sma7  = market["sp500_close"].rolling(7).mean()
_sma30 = market["sp500_close"].rolling(30).mean()
market["sp500_sma_cross_7_30"] = np.sign(_sma7 - _sma30)  # +1 bullish, -1 bearish, 0 neutral

# S&P 500 — Rate of Change
market["sp500_return_accel"]  = market["sp500_return_1d"].diff()    # 2nd-order: Δ(return)
market["sp500_vol_change_7d"] = market["sp500_vol_7d"].diff(5)      # week-over-week (5 trading days)

# S&P 500 — Lag Features
market["sp500_return_lag1"] = market["sp500_return_1d"].shift(1)
market["sp500_return_lag2"] = market["sp500_return_1d"].shift(2)
market["sp500_return_lag3"] = market["sp500_return_1d"].shift(3)

# Oil — Returns & Momentum
market["oil_return_1d"]       = market["oil_close"].pct_change(1)  * 100
market["oil_return_7d"]       = market["oil_close"].pct_change(7)  * 100
market["oil_return_14d"]      = market["oil_close"].pct_change(14) * 100
market["oil_rolling_mean_7d"] = market["oil_return_1d"].rolling(7).mean()

# Oil — Volatility
market["oil_vol_7d"]  = market["oil_return_1d"].rolling(7).std()
market["oil_vol_14d"] = market["oil_return_1d"].rolling(14).std()

# Oil — Level-Based
market["oil_dist_from_high_30d"] = (
    (market["oil_close"] - market["oil_close"].rolling(30).max())
    / market["oil_close"].rolling(30).max() * 100
)
market["oil_zscore_30d"] = (
    (market["oil_close"] - market["oil_close"].rolling(30).mean())
    / market["oil_close"].rolling(30).std()
)

# Oil — Lag Features
market["oil_return_lag1"] = market["oil_return_1d"].shift(1)
market["oil_return_lag2"] = market["oil_return_1d"].shift(2)

# ─── Reindex to full daily range and forward-fill weekends/holidays ───
# Features computed above on trading days → ffill propagates last known value to weekends
full_range = pd.DataFrame({"date": pd.date_range(market["date"].min(), market["date"].max())})
market = full_range.merge(market, on="date", how="left").ffill()

# ─── Merge into basetable ───
market_cols = [
    "date",
    "sp500_close",
    "sp500_return_1d", "sp500_return_3d", "sp500_return_7d", "sp500_return_14d",
    "sp500_rolling_mean_7d", "sp500_rolling_mean_14d",
    "sp500_vol_7d", "sp500_vol_14d", "sp500_vol_30d", "sp500_range_7d",
    "sp500_dist_from_high_30d", "sp500_dist_from_low_30d",
    "sp500_zscore_30d", "sp500_sma_cross_7_30",
    "sp500_return_accel", "sp500_vol_change_7d",
    "sp500_return_lag1", "sp500_return_lag2", "sp500_return_lag3",
    "oil_close",
    "oil_return_1d", "oil_return_7d", "oil_return_14d", "oil_rolling_mean_7d",
    "oil_vol_7d", "oil_vol_14d",
    "oil_dist_from_high_30d", "oil_zscore_30d",
    "oil_return_lag1", "oil_return_lag2",
    "vix_close", "bond_10y", "usd_index",
]
basetable = basetable.merge(market[market_cols], on="date", how="left")

## Macro data (monthly)
Published once a month → forward-fill to each day until the next release.
This correctly reflects what was *known* on each day (no lookahead bias).

In [13]:
# ─── Load macro data ───
macros = pd.read_csv("../Data/1_Bronze/Financials/macros.csv", parse_dates=["DATE"])
macros = macros.rename(columns={"DATE": "date"})
macros = macros.sort_values("date").reset_index(drop=True)

# ─── Forward-fill monthly values to daily ───
# Merge on full daily range, then ffill → each day gets the last known macro value
full_range = pd.DataFrame({"date": pd.date_range(basetable["date"].min(), basetable["date"].max())})
macros_daily = full_range.merge(macros, on="date", how="left").ffill()

# RealGDP is quarterly → many NaN even after ffill at start; ffill handles it
# Drop rows still NaN at the very start (before first macro release)
macros_daily = macros_daily.rename(columns={
    "RealDisposableIncome": "macro_real_income",
    "RealGDP":              "macro_real_gdp",
    "UnemploymentRate":     "macro_unemployment",
    "CPIInflation":         "macro_cpi",
    "ConsumerSentiment":    "macro_consumer_sentiment",
})

# Merge into basetable
basetable = basetable.merge(macros_daily, on="date", how="left")

basetable[["date", "macro_unemployment", "macro_cpi", "macro_consumer_sentiment"]].head(10)

,date,macro_unemployment,macro_cpi,macro_consumer_sentiment
0,2024-07-04,NaN,NaN,NaN
1,2024-07-05,NaN,NaN,NaN
2,2024-07-06,NaN,NaN,NaN
3,2024-07-07,NaN,NaN,NaN
4,2024-07-08,NaN,NaN,NaN
5,2024-07-09,NaN,NaN,NaN
6,2024-07-10,NaN,NaN,NaN
7,2024-07-11,NaN,NaN,NaN
8,2024-07-12,NaN,NaN,NaN
9,2024-07-13,NaN,NaN,NaN


# 6. Political Data

## Events

In [14]:
# ─── debate_day: presidential debate dates ───
# June 27 = Biden-Trump (CNN) — outside our data range (starts July 5)
# September 10 = Harris-Trump (ABC)
basetable["debate_day"]      = basetable["date"].isin(pd.to_datetime(["2024-09-10"])).astype(int)
basetable["debate_day_lag1"] = basetable["debate_day"].shift(1).fillna(0).astype(int)
basetable["debate_day_lag2"] = basetable["debate_day"].shift(2).fillna(0).astype(int)

# ─── Event dummies: one per event ───
def event_dummy(dates, lag=0):
    flags = basetable["date"].isin(pd.to_datetime(dates)).astype(int)
    return flags.shift(lag).fillna(0).astype(int)

# Assassination attempt 1 — July 13 (Butler, PA)
basetable["event_assassination1"]      = event_dummy(["2024-07-13"])
basetable["event_assassination1_lag1"] = event_dummy(["2024-07-13"], lag=1)

# Biden drops out — July 21
basetable["event_biden_out"]      = event_dummy(["2024-07-21"])
basetable["event_biden_out_lag1"] = event_dummy(["2024-07-21"], lag=1)

# Harris nominated — August 19
basetable["event_harris_nom"]      = event_dummy(["2024-08-19"])
basetable["event_harris_nom_lag1"] = event_dummy(["2024-08-19"], lag=1)

# Assassination attempt 2 — September 15
basetable["event_assassination2"]      = event_dummy(["2024-09-15"])
basetable["event_assassination2_lag1"] = event_dummy(["2024-09-15"], lag=1)

# Preview: only rows where something happened
event_cols = [c for c in basetable.columns if c.startswith("event_") or c.startswith("debate_")]
basetable[["date"] + event_cols].loc[basetable[event_cols].sum(axis=1) > 0]

,date,debate_day,debate_day_lag1,debate_day_lag2,event_assassination1,event_assassination1_lag1,event_biden_out,event_biden_out_lag1,event_harris_nom,event_harris_nom_lag1,event_assassination2,event_assassination2_lag1
9,2024-07-13,0,0,0,1,0,0,0,0,0,0,0
10,2024-07-14,0,0,0,0,1,0,0,0,0,0,0
17,2024-07-21,0,0,0,0,0,1,0,0,0,0,0
18,2024-07-22,0,0,0,0,0,0,1,0,0,0,0
46,2024-08-19,0,0,0,0,0,0,0,1,0,0,0
47,2024-08-20,0,0,0,0,0,0,0,0,1,0,0
68,2024-09-10,1,0,0,0,0,0,0,0,0,0,0
69,2024-09-11,0,1,0,0,0,0,0,0,0,0,0
70,2024-09-12,0,0,1,0,0,0,0,0,0,0,0
73,2024-09-15,0,0,0,0,0,0,0,0,0,1,0


## Polls

In [15]:
# --- Polls (wikipedia_polls.csv) ---------------------------------------------
# Polls are released irregularly (multiple per day or gaps of several days).
# Strategy: average same-day polls → forward-fill to get a daily value that
# reflects the LATEST known poll on each day (no lookahead bias).

polls = pd.read_csv("../Data/1_Bronze/Polls/wikipedia_polls.csv")
polls["Date"] = pd.to_datetime(polls["Date"])
polls["Margin"] = polls["Trump"] - polls["Harris"]  # recompute (64 NaN in source)

# Average multiple polls released on the same day
polls_daily = (polls.groupby("Date")
               .agg(poll_trump_avg   = ("Trump",  "mean"),
                    poll_harris_avg  = ("Harris", "mean"),
                    poll_margin      = ("Margin", "mean"),
                    poll_n_today     = ("Trump",  "count"))
               .reset_index()
               .rename(columns={"Date": "date"}))

# Reindex to full daily range and forward-fill gaps between polls
full_range      = pd.DataFrame({"date": pd.date_range(basetable["date"].min(),
                                                       basetable["date"].max())})
polls_daily_ff  = full_range.merge(polls_daily, on="date", how="left")
polls_daily_ff[["poll_trump_avg", "poll_harris_avg", "poll_margin"]] = (
    polls_daily_ff[["poll_trump_avg", "poll_harris_avg", "poll_margin"]].ffill())
polls_daily_ff["poll_n_today"] = polls_daily_ff["poll_n_today"].fillna(0).astype(int)

# 7-day rolling mean on forward-filled Trump avg (momentum signal)
polls_daily_ff["poll_trump_7d_avg"] = (
    polls_daily_ff["poll_trump_avg"].rolling(7, min_periods=1).mean())

# Merge into basetable
POLL_COLS = ["poll_trump_avg", "poll_harris_avg", "poll_margin",
             "poll_n_today", "poll_trump_7d_avg"]

basetable = basetable.merge(polls_daily_ff[["date"] + POLL_COLS], on="date", how="left")

print(f"Polls loaded: {len(polls)} raw entries, {len(polls_daily)} unique days")
basetable[["date"] + POLL_COLS].dropna(subset=["poll_trump_avg"]).head(8)

Polls loaded: 256 raw entries, 93 unique days


,date,poll_trump_avg,poll_harris_avg,poll_margin,poll_n_today,poll_trump_7d_avg
2,2024-07-06,41.000000,42.0,-1.000000,1,41.000000
3,2024-07-07,41.000000,42.0,-1.000000,0,41.000000
4,2024-07-08,49.000000,43.0,6.000000,1,43.666667
5,2024-07-09,45.333333,44.0,1.333333,3,44.083333
6,2024-07-10,49.000000,49.0,0.000000,2,45.066667
7,2024-07-11,49.000000,49.0,0.000000,0,45.722222
8,2024-07-12,49.000000,49.0,0.000000,0,46.190476
9,2024-07-13,49.000000,49.0,0.000000,0,47.333333


# 7. Lag Features (t-1)

In [16]:
# ─── Lag features (dag t-1) ───────────────────────────────────────────────────
# Waarom lags? Features zijn van dag t, target is einde-dag t prijs.
# Een lag geeft het model ook gisteren's waarde → detecteert momentum/trendbreuk.
#
# Lags worden toegevoegd voor:
#   • Bluesky sentiment/volume  — sentimentmomentum (stijgt/daalt het?)
#   • Nieuws volume/sentiment   — mediadruk opbouwend of afnemend?
#   • Markt (VIX, SP500)        — gisteren's angst/return als context

LAG_COLS = {
    # Bluesky
    "bsky_trump_post_share":    "bsky_trump_post_share_lag1",
    "bsky_trump_sentiment_avg": "bsky_trump_sentiment_avg_lag1",
    "bsky_post_count":          "bsky_post_count_lag1",
    # Nieuws
    "news_title_trump_count":    "news_title_trump_count_lag1",
    "news_attention_asymmetry":  "news_attention_asymmetry_lag1",
    "news_trump_sentiment_avg":  "news_trump_sentiment_avg_lag1",
    # Markt
    "vix_close":        "vix_close_lag1",
    "sp500_return_1d":  "sp500_return_lag1",
}

for src, dst in LAG_COLS.items():
    basetable[dst] = basetable[src].shift(1)

lag_cols = list(LAG_COLS.values())
print(f"{len(lag_cols)} lag-features aangemaakt")
basetable[["date"] + lag_cols].head(5)

8 lag-features aangemaakt


,date,bsky_trump_post_share_lag1,bsky_trump_sentiment_avg_lag1,bsky_post_count_lag1,news_title_trump_count_lag1,news_attention_asymmetry_lag1,news_trump_sentiment_avg_lag1,vix_close_lag1,sp500_return_lag1
0,2024-07-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2024-07-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2024-07-06,0.361111,-0.096700,36.0,851.0,592.0,-0.036589,12.48,NaN
3,2024-07-07,0.434783,0.094200,23.0,456.0,346.0,-0.074378,12.48,NaN
4,2024-07-08,0.481481,-0.297446,27.0,421.0,230.0,-0.038358,12.48,NaN


In [17]:
print(f"Basetable shape: {basetable.shape}")
print()

sections = {
    "Target": [
        "polymarket_trump_prob",
    ],
    "Autoregressive": [
        "polymarket_trump_prob_lag1",
    ],
    "Time": [
        "days_to_election", "week_nr", "weekend",
    ],
    "Bluesky — Volume": [
        "bsky_post_count", "bsky_trump_post_share",
        "bsky_reply_ratio", "bsky_repost_ratio",
    ],
    "Bluesky — Sentiment": [
        "bsky_trump_sentiment_avg",    "bsky_harris_sentiment_avg",
        "bsky_trump_sent_std",         "bsky_harris_sent_std",
        "bsky_trump_strong_pos_ratio", "bsky_harris_strong_pos_ratio",
        "bsky_trump_strong_neg_ratio", "bsky_harris_strong_neg_ratio",
        "bsky_sentiment_velocity",
    ],
    "Bluesky — Network": [
        "net_density", "net_trump_hub_degree_avg", "net_community_homophily",
        "net_bridge_post_share", "echo_chamber_velocity", "net_modularity_Q",
    ],
    "News — Volume": [
        "news_title_trump_count", "news_attention_asymmetry",
    ],
    "News — Sentiment (VADER)": [
        "news_trump_sentiment_avg",    "news_trump_sent_std",
        "news_trump_neg_ratio",        "news_trump_strong_pos_ratio",  "news_trump_strong_neg_ratio",
        "news_harris_sentiment_avg",   "news_harris_sent_std",
        "news_harris_neg_ratio",       "news_harris_strong_pos_ratio", "news_harris_strong_neg_ratio",
    ],
    "News — Sentiment (FinBERT)": [
        "news_trump_finbert_avg",        "news_trump_finbert_std",
        "news_trump_finbert_pos_ratio",  "news_trump_finbert_neg_ratio",
        "news_harris_finbert_avg",       "news_harris_finbert_std",
        "news_harris_finbert_pos_ratio", "news_harris_finbert_neg_ratio",
        "news_finbert_gap",
    ],
    "News — Topics": [
        "topic_economy_share", "topic_immigration_share",
    ],
    "Financials — Market": [
        "sp500_close", "sp500_return_1d", "oil_close",
        "vix_close", "bond_10y", "usd_index",
    ],
    "Financials — Macro": [
        "macro_real_income", "macro_real_gdp", "macro_unemployment",
        "macro_cpi", "macro_consumer_sentiment",
    ],
    "Polls": [
        "poll_trump_avg", "poll_harris_avg", "poll_margin",
        "poll_n_today", "poll_trump_7d_avg",
    ],
    "Political Events": [
        "debate_day", "debate_day_lag1", "debate_day_lag2",
        "event_assassination1", "event_assassination1_lag1",
        "event_biden_out", "event_biden_out_lag1",
        "event_harris_nom", "event_harris_nom_lag1",
        "event_assassination2", "event_assassination2_lag1",
    ],
    "Reddit — Volume": [
        "reddit_post_count", "reddit_trump_post_share", "reddit_harris_post_share",
        "reddit_trump_upvote_ratio", "reddit_comment_count",
        "reddit_conservative_activity", "reddit_cross_sub_ratio",
    ],
    "Reddit — Sentiment (VADER)": [
        "reddit_trump_sentiment_avg",    "reddit_trump_sent_std",
        "reddit_trump_strong_pos_ratio", "reddit_trump_strong_neg_ratio",
        "reddit_harris_sentiment_avg",   "reddit_harris_sent_std",
        "reddit_harris_strong_pos_ratio","reddit_harris_strong_neg_ratio",
        "reddit_sentiment_gap", "reddit_sentiment_platform_gap",
    ],
    "Lag features (t-1)": [
        "bsky_trump_post_share_lag1", "bsky_trump_sentiment_avg_lag1", "bsky_post_count_lag1",
        "news_title_trump_count_lag1", "news_attention_asymmetry_lag1", "news_trump_sentiment_avg_lag1",
        "vix_close_lag1", "sp500_return_lag1",
    ],
}

all_expected = [col for cols in sections.values() for col in cols]
missing = [c for c in all_expected if c not in basetable.columns]
extra   = [c for c in basetable.columns if c not in all_expected and c != "date"]

for section, cols in sections.items():
    present = [c for c in cols if c in basetable.columns]
    print(f"  [{len(present):>2}/{len(cols)}] {section}")
    for c in cols:
        flag = "✓" if c in basetable.columns else "✗ MISSING"
        print(f"         {flag}  {c}")
    print()

if missing:
    print(f"MISSING columns ({len(missing)}): {missing}")
if extra:
    print(f"Unlisted columns ({len(extra)}): {extra}")

basetable

Basetable shape: (124, 129)

  [ 1/1] Target
         ✓  polymarket_trump_prob

  [ 1/1] Autoregressive
         ✓  polymarket_trump_prob_lag1

  [ 3/3] Time
         ✓  days_to_election
         ✓  week_nr
         ✓  weekend

  [ 4/4] Bluesky — Volume
         ✓  bsky_post_count
         ✓  bsky_trump_post_share
         ✓  bsky_reply_ratio
         ✓  bsky_repost_ratio

  [ 9/9] Bluesky — Sentiment
         ✓  bsky_trump_sentiment_avg
         ✓  bsky_harris_sentiment_avg
         ✓  bsky_trump_sent_std
         ✓  bsky_harris_sent_std
         ✓  bsky_trump_strong_pos_ratio
         ✓  bsky_harris_strong_pos_ratio
         ✓  bsky_trump_strong_neg_ratio
         ✓  bsky_harris_strong_neg_ratio
         ✓  bsky_sentiment_velocity

  [ 6/6] Bluesky — Network
         ✓  net_density
         ✓  net_trump_hub_degree_avg
         ✓  net_community_homophily
         ✓  net_bridge_post_share
         ✓  echo_chamber_velocity
         ✓  net_modularity_Q

  [ 2/2] News — Volume
         ✓ 

,date,polymarket_trump_prob,polymarket_trump_prob_lag1,days_to_election,week_nr,weekend,bsky_post_count,bsky_trump_post_share,bsky_reply_ratio,bsky_repost_ratio,...,poll_margin,poll_n_today,poll_trump_7d_avg,bsky_trump_post_share_lag1,bsky_trump_sentiment_avg_lag1,bsky_post_count_lag1,news_title_trump_count_lag1,news_attention_asymmetry_lag1,news_trump_sentiment_avg_lag1,vix_close_lag1
0,2024-07-04,0.6050,NaN,124,27,0,NaN,NaN,NaN,NaN,...,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2024-07-05,0.6250,0.6050,123,27,0,36.0,0.361111,0.472222,0.222222,...,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2024-07-06,0.6250,0.6250,122,27,1,23.0,0.434783,0.130435,0.130435,...,-1.000000,1,41.000000,0.361111,-0.096700,36.0,851.0,592.0,-0.036589,12.480000
3,2024-07-07,0.6050,0.6250,121,27,1,27.0,0.481481,0.296296,0.148148,...,-1.000000,0,41.000000,0.434783,0.094200,23.0,456.0,346.0,-0.074378,12.480000
4,2024-07-08,0.6250,0.6050,120,28,0,22.0,0.590909,0.000000,0.454545,...,6.000000,1,43.666667,0.481481,-0.297446,27.0,421.0,230.0,-0.038358,12.480000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119,2024-10-31,0.5825,0.6325,5,44,0,689.0,0.509434,0.326560,0.301887,...,0.283333,6,47.535714,0.464706,-0.125703,680.0,2037.0,362.0,-0.012197,20.350000
120,2024-11-01,0.5585,0.5825,4,44,0,764.0,0.506545,0.297120,0.294503,...,-3.000000,1,47.250000,0.509434,-0.063054,689.0,1888.0,350.0,-0.013429,23.160000
121,2024-11-02,0.5635,0.5585,3,44,1,668.0,0.477545,0.326347,0.247006,...,-0.125000,8,47.510714,0.506545,-0.173601,764.0,1845.0,281.0,-0.068380,21.879999
122,2024-11-03,0.5855,0.5635,2,44,1,875.0,0.368000,0.342857,0.270857,...,-1.250000,6,47.648810,0.477545,-0.029818,668.0,1051.0,125.0,-0.009449,NaN


In [18]:
# ─── Save basetable → Data/3_Gold ─────────────────────────────────────────────
OUT_PATH = "../Data/3_Gold/basetable.csv"
basetable.to_csv(OUT_PATH, index=False)
print(f"Saved: {OUT_PATH}  ({basetable.shape[0]} rows x {basetable.shape[1]} cols)")

Saved: ../Data/3_Gold/basetable.csv  (124 rows x 129 cols)
